In [1]:
%%file app.py

from flask import Flask, request, jsonify

app = Flask(__name__)

counters = {
    "total": 0,
    "high": 0
}

def score_transaction(tx):
    score = 0
    rules = []
    if tx.get("amount", 0) > 3000:
        score += 3; rules.append("R1: kwota > 3000")
    if tx.get("category") == "elektronika" and tx.get("amount", 0) > 1500:
        score += 2; rules.append("R2: elektronika > 1500")
    if tx.get("hour", 12) < 6:
        score += 2; rules.append("R3: nocna godzina")
    risk_level = "CRITICAL" if score >= 5 else "HIGH" if score >= 3 else "MEDIUM" if score >= 1 else "LOW"
    return {"score": score, "risk_level": risk_level, "triggered_rules": rules}


@app.route("/score", methods=["POST"])
def score():
    tx = request.get_json()
    if not tx or "amount" not in tx:
        return jsonify({
            "error": "Brak pola 'amount'"
        }), 400
# Zadanie 2    
    if tx["amount"] < 0:
        return jsonify({
            "error": "Amount nie może być ujemny"
        }), 400
        
    counters["total"] += 1
    result = score_transaction(tx)
    if result["risk_level"] in ["HIGH", "CRITICAL"]:
        counters["high"] += 1
    result["tx_id"] = tx.get("tx_id", "unknown")
    return jsonify(result)
    
# Zadanie 1
@app.route("/stats", methods=["GET"])
def stats():
    return jsonify({
        "total_requests": counters["total"],
        "high_or_critical": counters["high"]
    })

@app.route("/health")
def health():
    return jsonify({"status": "ok", "version": "1.0-rules"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [2]:
import subprocess
import time

server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

print("Serwer uruchomiony")

Serwer uruchomiony


In [3]:
# Zadanie 3
import requests

transactions = [
    {"tx_id": "TX001", "amount": 50,    "category": "żywność",      "hour": 12},
    {"tx_id": "TX002", "amount": 6000,  "category": "elektronika",  "hour": 2},
    {"tx_id": "TX003", "amount": 1700,  "category": "elektronika",  "hour": 15},
    {"tx_id": "TX004", "amount": 3200,  "category": "sport",        "hour": 18},
    {"tx_id": "TX005", "amount": 100,   "category": "żywność",      "hour": 1},
    {"tx_id": "TX006", "amount": 4500,  "category": "elektronika",  "hour": 4},
    {"tx_id": "TX007", "amount": 900,   "category": "moda",         "hour": 14},
    {"tx_id": "TX008", "amount": 2800,  "category": "elektronika",  "hour": 5},
    {"tx_id": "TX009", "amount": 7000,  "category": "elektronika",  "hour": 3},
    {"tx_id": "TX010", "amount": -50,   "category": "test",         "hour": 10}
]
print(f"{'TX_ID':<8} {'AMOUNT':<10} {'RISK':<10} {'SCORE':<8}")

for tx in transactions:
    r = requests.post(
        "http://localhost:5000/score",
        json=tx
    )
    
    if r.status_code != 200:
        print(
            f"{tx['tx_id']:<8} "
            f"ERROR -> "
            f"{r.json()['error']}"
        )
        continue 

    res = r.json()
    print(
        f"{tx['tx_id']:<8} "
        f"{tx['amount']:<10} "
        f"{res['risk_level']:<10} "
        f"{res['score']:<8} "
    )

TX_ID    AMOUNT     RISK       SCORE   
TX001    50         LOW        0        
TX002    6000       CRITICAL   7        
TX003    1700       MEDIUM     2        
TX004    3200       HIGH       3        
TX005    100        MEDIUM     2        
TX006    4500       CRITICAL   7        
TX007    900        LOW        0        
TX008    2800       HIGH       4        
TX009    7000       CRITICAL   7        
TX010    ERROR -> Amount nie może być ujemny


In [4]:
#Zadaie 1 - wynik
stats = requests.get("http://localhost:5000/stats")
print(stats.json())

{'high_or_critical': 5, 'total_requests': 9}


In [5]:
server.kill()
print("Serwer zatrzymany.")

Serwer zatrzymany.
